# Notebook 08 — T2 Task: Lorenz Attractor

**Task (T2):** One-step-ahead prediction on the Lorenz attractor.
Given `(x_t, y_t, z_t)`, predict `x_{t+1}`.

**Why Lorenz?**
- Chaotic dynamics require rich non-linear function approximation
- Classical MLP with few parameters fails; tests whether quantum methods
  gain an advantage from their non-linearity mechanism
- Directly connects to the "edge of chaos" theme in PIPELINE.md
  (Kobayashi & Motome 2025; edge-of-chaos in QRC)

**Encoding strategy:** 3 inputs `(x_t, y_t, z_t)` mapped to 3 qubit angles;
circuits extended to multi-input encoding.

**Metrics:** `R²_test`, a continuous TER score derived from train-curve area,
best train score, test score, and parameter efficiency, plus short-horizon
trajectory fidelity.

In [ ]:
import sys
sys.path.insert(0, "..")

import math
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.metrics import r2_score
from sklearn.preprocessing import MinMaxScaler
import json, pathlib, os

from src.qnlep_registry import param_shape
from src.circuits.lorenz_qnlep import build_lorenz_registry
from src.qnlep_run_paths import protocol_results_dir, protocol_n_qubits, protocol_n_layers
from src.qnlep_figures import repo_figures_dir, safe_savefig

FIGURES_DIR = repo_figures_dir()
from src.metrics.protocol_scores import trainability_efficiency_score, shifted_r2_score

RUN_SEED = int(os.environ.get("QNLEP_SEED", "42"))
np.random.seed(RUN_SEED)
print("QNLEP_SEED =", RUN_SEED)


## 1. Generate Lorenz attractor data

In [ ]:
# Lorenz parameters
SIGMA, RHO, BETA = 10.0, 28.0, 8.0 / 3.0
DT               = 0.01
N_STEPS_TOTAL    = 3000    # integrate for 3000 steps; discard first 500 (transient)
TRANSIENT        = 500

def lorenz_step(state, dt=DT):
    x, y, z = state
    dx = SIGMA * (y - x)
    dy = x * (RHO - z) - y
    dz = x * y - BETA * z
    return np.array([x + dx * dt, y + dy * dt, z + dz * dt])

# Integrate
trajectory = [np.array([1.0, 1.0, 1.0])]
for _ in range(N_STEPS_TOTAL - 1):
    trajectory.append(lorenz_step(trajectory[-1]))
trajectory = np.array(trajectory)[TRANSIENT:]   # shape: (2500, 3)

# Normalise to [-π, π] for angle encoding
scaler = MinMaxScaler(feature_range=(-np.pi, np.pi))
traj_scaled = np.asarray(scaler.fit_transform(trajectory), dtype=float)

print(f"Trajectory shape after transient removal: {trajectory.shape}")
print(f"x range (scaled): [{traj_scaled[:, 0].min():.2f}, {traj_scaled[:, 0].max():.2f}]")


### 1a. Visualize the attractor

In [ ]:
fig = plt.figure(figsize=(8, 5))
ax  = fig.add_subplot(111, projection='3d')
ax.plot(trajectory[:, 0], trajectory[:, 1], trajectory[:, 2],
        lw=0.3, alpha=0.6, color='steelblue')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title(f'Lorenz attractor  (σ={SIGMA}, ρ={RHO}, β={BETA:.2f})')
plt.tight_layout()
safe_savefig(FIGURES_DIR / 'lorenz_attractor.pdf', bbox_inches='tight')
plt.show()


## 2. Prepare train / test split

In [ ]:
N_TRAIN = 120   # one-step dataset (larger = stabler R² / TER; keep start+N_TRAIN+N_TEST+N_ROLLOUT within trajectory)
N_TEST  = 60

# Input: state at t; Target: x-coordinate at t+1
X_all = traj_scaled[:-1]   # (2499, 3)
Y_all = traj_scaled[1:, 0] # (2499,) — predict next x only

# Sample a contiguous training window from the middle of the attractor
start  = 500
X_train = X_all[start : start + N_TRAIN]   # (N_TRAIN, 3)
Y_train = Y_all[start : start + N_TRAIN]

X_test  = X_all[start + N_TRAIN : start + N_TRAIN + N_TEST]
Y_test  = Y_all[start + N_TRAIN : start + N_TRAIN + N_TEST]

_need = start + N_TRAIN + N_TEST + 80  # headroom for rollout horizon in §5c
assert _need <= len(traj_scaled), (
    "Trajectory too short for start, N_TRAIN, N_TEST, rollout — increase N_STEPS_TOTAL or shrink window."
)

print(f"Training samples: {N_TRAIN}  Test samples: {N_TEST}")


## 3. Multi-input circuit extensions

Original circuits take a scalar `x`. For 3-input Lorenz, we extend the encoding
to distribute `(x_t, y_t, z_t)` across qubits.

This notebook now builds the **full QNLEP registry** for Lorenz one-step prediction,
so `ter_scores.json` covers every method in the protocol with the same task setup.

In [ ]:
N_QUBITS = protocol_n_qubits(3)
N_LAYERS = protocol_n_layers(3)
N_STEPS  = 200   # larger optimization budget for the full-registry Lorenz comparison
LR       = 0.1
TER_EVAL_EVERY = 10

CIRCUITS = build_lorenz_registry(N_QUBITS, N_LAYERS, run_seed=RUN_SEED)
print('Lorenz TER methods:', list(CIRCUITS))


## 4. Train each method and compute TER on Lorenz

In [ ]:
lorenz_results = {}

for method_name, circuit in CIRCUITS.items():
    shape = param_shape(method_name, N_QUBITS, N_LAYERS)
    weights = pnp.array(np.random.normal(0, 0.1, shape), requires_grad=True)

    def cost(w):
        preds = pnp.array([circuit(w, state) for state in X_train])
        return pnp.mean((preds - Y_train) ** 2)

    opt = qml.AdamOptimizer(stepsize=LR)
    history = []
    eval_steps = []
    train_r2_curve = []
    train_score_curve = []

    for step in range(N_STEPS):
        weights, loss = opt.step_and_cost(cost, weights)
        history.append(float(loss))

        should_eval = ((step + 1) % TER_EVAL_EVERY == 0) or (step == N_STEPS - 1)
        if should_eval:
            preds_train = np.array([float(circuit(weights, s)) for s in X_train])
            train_r2 = float(r2_score(Y_train, preds_train))
            train_score = shifted_r2_score(train_r2)
            eval_steps.append(step + 1)
            train_r2_curve.append(train_r2)
            train_score_curve.append(train_score)

    preds_test = np.array([float(circuit(weights, s)) for s in X_test])
    r2_test = float(r2_score(Y_test, preds_test))
    n_params = int(np.asarray(weights).size)

    lorenz_results[method_name] = {
        'r2_test': r2_test,
        'n_params': n_params,
        'history': history,
        'eval_steps': eval_steps,
        'train_r2_curve': train_r2_curve,
        'train_loss_curve': history,
        'train_score_curve': train_score_curve,
        'best_train_r2': float(max(train_r2_curve)) if train_r2_curve else np.nan,
        'final_train_r2': float(train_r2_curve[-1]) if train_r2_curve else np.nan,
        'params': np.asarray(weights, dtype=float),
        'preds_test': preds_test.tolist(),
    }

min_params = min(lorenz_results[m]['n_params'] for m in CIRCUITS)
for method_name in CIRCUITS:
    comp = trainability_efficiency_score(
        lorenz_results[method_name]['train_score_curve'],
        lorenz_results[method_name]['r2_test'],
        lorenz_results[method_name]['n_params'],
        min_params,
    )
    lorenz_results[method_name].update(comp)
    lorenz_results[method_name]['ter'] = lorenz_results[method_name]['ter_raw']
    print(
        f"{method_name:20s}  best_train_R²={lorenz_results[method_name]['best_train_r2']:.3f}  "
        f"test_R²={lorenz_results[method_name]['r2_test']:.3f}  TER={lorenz_results[method_name]['ter']:.3f}"
    )


## 5. Visualizations

### 5a. Training loss curves

In [ ]:
RESULTS_DIR = protocol_results_dir()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
export = {
    m: {
        k: v for k, v in lorenz_results[m].items()
        if k not in ('history', 'params')
    }
    for m in CIRCUITS
}

ter_scores = {m: float(lorenz_results[m]['ter']) for m in CIRCUITS}

with open(RESULTS_DIR / 'ter_scores.json', 'w', encoding='utf-8') as f:
    json.dump(ter_scores, f, indent=2)
with open(RESULTS_DIR / 'lorenz_results.json', 'w', encoding='utf-8') as f:
    json.dump(export, f, indent=2)
meta = {
    'score_definition': 'continuous_trainability_efficiency_score',
    'ter_trained_methods': list(CIRCUITS.keys()),
    'n_train': N_TRAIN,
    'n_test': N_TEST,
    'n_steps': N_STEPS,
    'train_eval_every': TER_EVAL_EVERY,
    'train_window_start': start,
    'run_seed': RUN_SEED,
    'n_qubits': N_QUBITS,
    'n_layers': N_LAYERS,
    'min_params_reference': min(lorenz_results[m]['n_params'] for m in CIRCUITS),
    'components': {
        'curve_score': 0.50,
        'best_train_score': 0.30,
        'test_score': 0.20,
        'param_efficiency': 'sqrt(min_params / n_params) multiplier',
    },
}
with open(RESULTS_DIR / 'ter_scores_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2)
ter_learning_curves = {
    m: {
        k: export[m][k]
        for k in ('eval_steps', 'train_r2_curve', 'train_score_curve', 'train_loss_curve')
        if k in export[m]
    }
    for m in CIRCUITS
}
with open(RESULTS_DIR / 'ter_learning_curves.json', 'w', encoding='utf-8') as f:
    json.dump(ter_learning_curves, f, indent=2)
print('Saved ter_scores.json, ter_scores_meta.json, lorenz_results.json')


In [ ]:
n_cols = min(3, len(CIRCUITS))
n_rows = int(math.ceil(len(CIRCUITS) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.2 * n_cols, 3.2 * n_rows), squeeze=False)
axes = axes.ravel()

for i, method_name in enumerate(CIRCUITS):
    ax = axes[i]
    ax.semilogy(lorenz_results[method_name]['history'],
                color=plt.cm.tab10.colors[i % 10], lw=1.5)
    ax.set_xlabel('Step')
    ax.set_ylabel('MSE (log)')
    ax.set_title(method_name.replace('_', ' '))
    ax.grid(True, alpha=0.3)

for ax in axes[len(CIRCUITS):]:
    ax.axis('off')

fig.suptitle('Lorenz one-step prediction — training curves', y=1.02)
plt.tight_layout()
safe_savefig(FIGURES_DIR / 'lorenz_training.pdf', bbox_inches='tight')
plt.show()


### 5b. Predicted vs true x(t+1) on test set

In [ ]:
n_cols = min(3, len(CIRCUITS))
n_rows = int(math.ceil(len(CIRCUITS) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.0 * n_cols, 3.4 * n_rows), squeeze=False)
axes = np.asarray(axes, dtype=object).ravel()
t_test = np.arange(N_TEST)

for i, method_name in enumerate(CIRCUITS):
    ax = axes[i]
    r = lorenz_results[method_name]
    ax.plot(t_test, Y_test, 'k--', lw=1.5, label='true')
    ax.plot(t_test, r['preds_test'], 'C0', lw=2.0,
            label=f"pred  R²={r['r2_test']:.2f}")
    ax.set_xlabel('Test step')
    ax.set_title(method_name.replace('_', ' '))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

for ax in axes[len(CIRCUITS):]:
    ax.axis('off')

axes[0].set_ylabel('x(t+1)  [scaled]')
fig.suptitle('Lorenz one-step prediction — test set', y=1.02)
plt.tight_layout()
safe_savefig(FIGURES_DIR / 'lorenz_predictions.pdf', bbox_inches='tight')
plt.show()


### 5c. Autoregressive rollout (short-horizon trajectory)

Feed predictions back as inputs for multi-step forecasting.
Error accumulation rate reflects sensitivity to non-linearity quality.

In [ ]:
N_ROLLOUT = 60

n_cols = min(3, len(CIRCUITS))
n_rows = int(math.ceil(len(CIRCUITS) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.0 * n_cols, 3.4 * n_rows), sharey=True, squeeze=False)
axes = np.asarray(axes, dtype=object).ravel()
traj_x = np.asarray(traj_scaled, dtype=float)[:, 0]
true_x = traj_x[start + N_TRAIN : start + N_TRAIN + N_ROLLOUT + 1]

for i, method_name in enumerate(CIRCUITS):
    ax = axes[i]
    circuit = CIRCUITS[method_name]
    params  = pnp.array(lorenz_results[method_name]['params'], requires_grad=False)

    # start rollout from ground-truth state at test start
    state     = X_test[0].copy()
    rollout_x = [state[0]]

    for t in range(N_ROLLOUT):
        pred_x = float(circuit(params, state))
        # update state: use predicted x, shift y->z (simple autoregressive proxy)
        state = np.array([pred_x, state[0], state[1]])
        rollout_x.append(pred_x)

    ax.plot(true_x, 'k--', lw=1.5, label='true')
    ax.plot(rollout_x, 'C0', lw=2.0, label='autoregressive')
    ax.set_xlabel('Rollout step')
    ax.set_title(method_name.replace('_', ' '))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

for ax in axes[len(CIRCUITS):]:
    ax.axis('off')

axes[0].set_ylabel('x(t)  [scaled]')
fig.suptitle('Autoregressive rollout — trajectory fidelity', y=1.02)
plt.tight_layout()
safe_savefig(FIGURES_DIR / 'lorenz_rollout.pdf', bbox_inches='tight')
plt.show()


## 6. Export results